# 🚀 Traffic Demand Forecasting — Momentum Pipeline
**Model:** CatBoost Regressor with Chronological Lag Features

| Cell | Description |
|------|-------------|
| 1 | Environment setup & data ingestion |
| 2 | Chronological sequence reconstruction & momentum generation |
| 3 | Leak-free spatial encoding & model training |
| 4 | Inference, inversion & submission export |

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT & FILE INGESTION
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import warnings, os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
np.random.seed(42)

# ── Load data ─────────────────────────────────────────────────────────────────
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"Train days  : {sorted(train_df['day'].unique())}")
print(f"Test days   : {sorted(test_df['day'].unique())}")
print(f"Train day49 timestamps: {sorted(train_df[train_df['day']==49]['timestamp'].unique())}")
print(f"Test timestamps range : {sorted(test_df['timestamp'].unique())[0]} → {sorted(test_df['timestamp'].unique())[-1]}")
print(f"Unique geohashes (train): {train_df['geohash'].nunique()}")
print(f"Unique geohashes (test) : {test_df['geohash'].nunique()}")
print(f"\nTarget stats — mean: {train_df['demand'].mean():.5f}, "
      f"median: {train_df['demand'].median():.5f}, skew: {train_df['demand'].skew():.2f}")
print("\n✅ Data loaded successfully")

Train shape : (77299, 11)
Test shape  : (41778, 10)
Train days  : [np.int64(48), np.int64(49)]
Test days   : [np.int64(49)]
Train day49 timestamps: ['0:0', '0:15', '0:30', '0:45', '1:0', '1:15', '1:30', '1:45', '2:0']
Test timestamps range : 10:0 → 9:45
Unique geohashes (train): 1249
Unique geohashes (test) : 1190

Target stats — mean: 0.09394, median: 0.04776, skew: 3.73

✅ Data loaded successfully


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — CHRONOLOGICAL SEQUENCE RECONSTRUCTION & MOMENTUM GENERATION
#
# KEY INSIGHT: The test set (day 49, 2:15→13:45) is the chronological
# continuation of the training data (day 48 full + day 49 0:00→2:00).
# We MUST concatenate before computing lags so that the first test rows
# inherit momentum from the last training rows for each geohash.
# ══════════════════════════════════════════════════════════════════════════════

# ── Mark train/test partitions ────────────────────────────────────────────────
train_df['is_test'] = 0
test_df['is_test']  = 1
test_df['demand']   = np.nan  # Placeholder; never used in training

# ── Unify column order and concatenate ────────────────────────────────────────
all_cols = [c for c in train_df.columns if c in test_df.columns] + \
           [c for c in train_df.columns if c not in test_df.columns]
combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
print(f"Combined shape: {combined.shape}")

# ── Parse timestamp → Hour, Minute ────────────────────────────────────────────
parts = combined['timestamp'].str.split(':', expand=True).astype(int)
combined['Hour']   = parts[0]
combined['Minute'] = parts[1]

# ── Construct continuous linear timeline (in minutes) ─────────────────────────
combined['total_minutes'] = combined['day'] * 24 * 60 + combined['Hour'] * 60 + combined['Minute']

# ── TimeSlot: 96 discrete slots per day (0–95) ────────────────────────────────
combined['TimeSlot'] = combined['Hour'] * 4 + combined['Minute'] // 15

# ── Rigorous chronological sort ───────────────────────────────────────────────
combined.sort_values(by=['geohash', 'total_minutes'], inplace=True)
combined.reset_index(drop=True, inplace=True)

# ── Impute missing contextual features BEFORE lag computation ─────────────────
temp_median = combined.loc[combined['is_test'] == 0, 'Temperature'].median()
combined['Temperature']   = combined['Temperature'].fillna(temp_median)
combined['RoadType']      = combined['RoadType'].fillna('Unknown')
combined['Weather']       = combined['Weather'].fillna('Unknown')
combined['LargeVehicles'] = combined['LargeVehicles'].fillna('Unknown')
combined['Landmarks']     = combined['Landmarks'].fillna('Unknown')

# ── Log-transform demand (only non-null = train rows) ─────────────────────────
combined['demand_log'] = np.log1p(combined['demand'])

# ══════════════════════════════════════════════════════════════════════════════
# MOMENTUM FEATURES: Lag & Rolling Window within each geohash's timeline
#
# Because we sorted chronologically and concatenated, the last training
# rows for geohash X flow directly into the first test rows for geohash X.
# Lag features for test row at 2:15 will correctly pull from train at 2:00.
# ══════════════════════════════════════════════════════════════════════════════

geo_groups = combined.groupby('geohash')['demand_log']

# Short-term lags: 1 step (15 min), 2 steps (30 min), 4 steps (1 hour)
combined['lag_1']  = geo_groups.shift(1)   # t-15min
combined['lag_2']  = geo_groups.shift(2)   # t-30min
combined['lag_4']  = geo_groups.shift(4)   # t-1hr

# Rolling mean over preceding 4 steps (= 1 hour window), min_periods=1
combined['roll_mean_4'] = geo_groups.transform(
    lambda s: s.shift(1).rolling(window=4, min_periods=1).mean()
)

# Rolling std over preceding 4 steps (volatility signal)
combined['roll_std_4'] = geo_groups.transform(
    lambda s: s.shift(1).rolling(window=4, min_periods=1).std()
)
combined['roll_std_4'] = combined['roll_std_4'].fillna(0.0)

# Momentum delta: difference between last value and the one before it
combined['momentum_1'] = combined['lag_1'] - combined['lag_2']

# Same-time-yesterday lag: for day 49, grab from the same TimeSlot on day 48
# This is 96 steps back in a fully sorted geohash timeline
combined['lag_96'] = geo_groups.shift(96)

# ── Cyclic time encodings (high-resolution sine/cosine) ───────────────────────
minutes_in_day = combined['Hour'] * 60 + combined['Minute']

combined['time_sin']     = np.sin(2 * np.pi * minutes_in_day / 1440.0)
combined['time_cos']     = np.cos(2 * np.pi * minutes_in_day / 1440.0)
combined['hour_sin']     = np.sin(2 * np.pi * combined['Hour'] / 24.0)
combined['hour_cos']     = np.cos(2 * np.pi * combined['Hour'] / 24.0)
combined['slot_sin']     = np.sin(2 * np.pi * combined['TimeSlot'] / 96.0)
combined['slot_cos']     = np.cos(2 * np.pi * combined['TimeSlot'] / 96.0)

# ── Binary period flags ───────────────────────────────────────────────────────
combined['IsRushHour'] = combined['Hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
combined['IsNight']    = combined['Hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
combined['IsMidDay']   = combined['Hour'].isin([11, 12, 13, 14]).astype(int)

# ── Geo prefix features (spatial neighborhood) ────────────────────────────────
combined['geo_prefix4'] = combined['geohash'].str[:4]
combined['geo_prefix5'] = combined['geohash'].str[:5]

# ── Report ────────────────────────────────────────────────────────────────────
lag_cols = ['lag_1', 'lag_2', 'lag_4', 'roll_mean_4', 'roll_std_4', 'momentum_1', 'lag_96']
test_mask = combined['is_test'] == 1

print(f"\nLag feature coverage on TEST rows:")
for c in lag_cols:
    pct = combined.loc[test_mask, c].notna().mean() * 100
    print(f"  {c:18s}: {pct:6.2f}% populated")

print(f"\n✅ Momentum features generated — {len(lag_cols)} sequential features created")

Combined shape: (119077, 12)

Lag feature coverage on TEST rows:
  lag_1             :   2.82% populated
  lag_2             :   5.57% populated
  lag_4             :  10.84% populated
  roll_mean_4       :  10.98% populated
  roll_std_4        : 100.00% populated
  momentum_1        :   2.80% populated
  lag_96            :  62.09% populated

✅ Momentum features generated — 7 sequential features created


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — LEAK-FREE SPATIAL ENCODING & MODEL TRAINING
# ══════════════════════════════════════════════════════════════════════════════

# ── Split back into train and test ────────────────────────────────────────────
train_full = combined[combined['is_test'] == 0].copy()
test_final = combined[combined['is_test'] == 1].copy()

# Preserve original Index for submission
test_index = test_final['Index'].values

# ── Define feature columns ────────────────────────────────────────────────────
CAT_FEATURES = ['geohash', 'RoadType', 'Weather', 'LargeVehicles',
                'Landmarks', 'geo_prefix4', 'geo_prefix5']

DROP_COLS = ['Index', 'timestamp', 'demand', 'demand_log', 'is_test', 'total_minutes']

FEATURE_COLS = [c for c in combined.columns if c not in DROP_COLS]
print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")

# Cast categoricals to string
for col in CAT_FEATURES:
    train_full[col] = train_full[col].astype(str)
    test_final[col] = test_final[col].astype(str)

# ── Target variable ───────────────────────────────────────────────────────────
y_all = train_full['demand_log'].values

# ══════════════════════════════════════════════════════════════════════════════
# LEAK-FREE TARGET ENCODING
#
# Encoding maps are computed ONLY on the training partition.
# For the chronological validation split, we use a temporal mask so the
# model evaluates on the future (day 49 training rows).
# ══════════════════════════════════════════════════════════════════════════════

overall_mean = np.nanmean(y_all)

def make_key(df, cols):
    return df[cols].astype(str).agg('_'.join, axis=1)

# Encoding specifications: feature_name → groupby_columns
ENC_SPECS = {
    'enc_geo':           ['geohash'],
    'enc_geo_hour':      ['geohash', 'Hour'],
    'enc_geo_slot':      ['geohash', 'TimeSlot'],
    'enc_prefix4':       ['geo_prefix4'],
    'enc_prefix5_hour':  ['geo_prefix5', 'Hour'],
}

# Build encoding maps from ALL training rows
enc_maps = {}
for feat_name, cols in ENC_SPECS.items():
    key = make_key(train_full, cols)
    enc_maps[feat_name] = train_full.groupby(key)['demand_log'].mean()

# Apply to both train and test
for feat_name, cols in ENC_SPECS.items():
    m = enc_maps[feat_name]
    # Train
    key_tr = make_key(train_full, cols)
    train_full[feat_name] = key_tr.map(m)
    # Test
    key_te = make_key(test_final, cols)
    test_final[feat_name] = key_te.map(m)

# Hierarchical fallback for NaN encodings
for feat_name in ENC_SPECS:
    if feat_name != 'enc_geo' and feat_name != 'enc_prefix4':
        # Fallback: geohash-level → prefix4 → overall mean
        train_full[feat_name] = train_full[feat_name].fillna(train_full['enc_geo'])
        test_final[feat_name] = test_final[feat_name].fillna(test_final['enc_geo'])
    if feat_name == 'enc_geo':
        # Fallback: prefix4 → overall mean
        train_full[feat_name] = train_full[feat_name].fillna(train_full['enc_prefix4'])
        test_final[feat_name] = test_final[feat_name].fillna(test_final['enc_prefix4'])
    train_full[feat_name] = train_full[feat_name].fillna(overall_mean)
    test_final[feat_name] = test_final[feat_name].fillna(overall_mean)

# Add encoding columns to feature list
enc_feat_names = list(ENC_SPECS.keys())
ALL_FEATURES = FEATURE_COLS + enc_feat_names

# Fill remaining NaN in lag features with the geohash encoding (reasonable prior)
for c in lag_cols:
    train_full[c] = train_full[c].fillna(train_full['enc_geo'])
    test_final[c] = test_final[c].fillna(test_final['enc_geo'])
    train_full[c] = train_full[c].fillna(overall_mean)
    test_final[c] = test_final[c].fillna(overall_mean)

print(f"\nFinal feature count: {len(ALL_FEATURES)}")
print(f"Encoding features: {enc_feat_names}")

# ══════════════════════════════════════════════════════════════════════════════
# CHRONOLOGICAL VALIDATION SPLIT
#
# Use day 49 training rows as validation (forward horizon).
# This simulates predicting the future and avoids shuffle leakage.
# ══════════════════════════════════════════════════════════════════════════════

chrono_train_mask = train_full['day'] == 48
chrono_val_mask   = train_full['day'] == 49

X_chrono_tr = train_full.loc[chrono_train_mask, ALL_FEATURES]
y_chrono_tr = train_full.loc[chrono_train_mask, 'demand_log'].values
X_chrono_vl = train_full.loc[chrono_val_mask, ALL_FEATURES]
y_chrono_vl = train_full.loc[chrono_val_mask, 'demand_log'].values

cat_idx = [ALL_FEATURES.index(c) for c in CAT_FEATURES]

print(f"\nChrono split — train: {len(X_chrono_tr)}, val: {len(X_chrono_vl)}")

train_pool_chrono = Pool(X_chrono_tr, y_chrono_tr, cat_features=cat_idx)
val_pool_chrono   = Pool(X_chrono_vl, y_chrono_vl, cat_features=cat_idx)

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1: VALIDATION RUN — Find optimal iteration count
# ══════════════════════════════════════════════════════════════════════════════

print("\n🔍 Running validation model to find best iteration...")

model_val = CatBoostRegressor(
    iterations        = 8000,
    learning_rate     = 0.03,
    depth             = 8,
    l2_leaf_reg       = 5,
    min_data_in_leaf  = 20,
    subsample         = 0.8,
    colsample_bylevel = 0.8,
    grow_policy       = 'SymmetricTree',
    random_seed       = 42,
    eval_metric       = 'RMSE',
    od_type           = 'Iter',
    od_wait           = 300,
    verbose           = 500,
    task_type         = 'CPU',
    bootstrap_type    = 'MVS',
)

model_val.fit(train_pool_chrono, eval_set=val_pool_chrono, use_best_model=True)

val_preds_log = model_val.predict(val_pool_chrono)
val_preds     = np.clip(np.expm1(val_preds_log), 0, None)
val_actual    = np.expm1(y_chrono_vl)

r2   = r2_score(val_actual, val_preds)
rmse = np.sqrt(mean_squared_error(val_actual, val_preds))
best_iter = model_val.get_best_iteration()

print(f"\n{'='*50}")
print(f"  CHRONOLOGICAL VALIDATION RESULTS")
print(f"  R² Score       : {r2*100:.4f}%")
print(f"  RMSE           : {rmse:.6f}")
print(f"  Best iteration : {best_iter}")
print(f"{'='*50}")

# ── Feature importance ────────────────────────────────────────────────────────
fi = pd.Series(model_val.get_feature_importance(), index=ALL_FEATURES)
print("\n📊 Top 15 features:")
print(fi.nlargest(15).to_string())

# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2: FINAL MODEL — Train on ALL training data
# ══════════════════════════════════════════════════════════════════════════════

final_iters = max(int(best_iter * 1.15), best_iter + 200)
print(f"\n🚀 Training FINAL model on ALL {len(train_full)} training rows ")
print(f"   for {final_iters} iterations (best={best_iter} + 15% buffer)...")

X_train_all = train_full[ALL_FEATURES]
y_train_all = y_all

train_pool_full = Pool(X_train_all, y_train_all, cat_features=cat_idx)

model_final = CatBoostRegressor(
    iterations        = final_iters,
    learning_rate     = 0.03,
    depth             = 8,
    l2_leaf_reg       = 5,
    min_data_in_leaf  = 20,
    subsample         = 0.8,
    colsample_bylevel = 0.8,
    grow_policy       = 'SymmetricTree',
    random_seed       = 42,
    eval_metric       = 'RMSE',
    verbose           = 500,
    task_type         = 'CPU',
    bootstrap_type    = 'MVS',
)

model_final.fit(train_pool_full)
print("\n✅ Final model trained successfully")

Feature columns (29): ['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'TimeSlot', 'lag_1', 'lag_2', 'lag_4', 'roll_mean_4', 'roll_std_4', 'momentum_1', 'lag_96', 'time_sin', 'time_cos', 'hour_sin', 'hour_cos', 'slot_sin', 'slot_cos', 'IsRushHour', 'IsNight', 'IsMidDay', 'geo_prefix4', 'geo_prefix5']

Final feature count: 34
Encoding features: ['enc_geo', 'enc_geo_hour', 'enc_geo_slot', 'enc_prefix4', 'enc_prefix5_hour']

Chrono split — train: 69427, val: 7872

🔍 Running validation model to find best iteration...
0:	learn: 0.1053736	test: 0.1103111	best: 0.1103111 (0)	total: 375ms	remaining: 49m 59s
500:	learn: 0.0057545	test: 0.0411572	best: 0.0411572 (500)	total: 2m 49s	remaining: 42m 29s
1000:	learn: 0.0049259	test: 0.0407007	best: 0.0406509 (926)	total: 5m 32s	remaining: 38m 48s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.04065088018
bestIteration = 926

Shrink model to first 927 itera

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — INVERSION & SANITIZED SUBMISSION OUTPUT
# ══════════════════════════════════════════════════════════════════════════════

# ── Inference on test set ─────────────────────────────────────────────────────
X_test_final = test_final[ALL_FEATURES]
test_pool    = Pool(X_test_final, cat_features=cat_idx)

test_preds_log = model_final.predict(test_pool)

# ── Reverse log1p transform and sanitize ──────────────────────────────────────
test_preds = np.clip(np.expm1(test_preds_log), 0, None)

# ── Build submission DataFrame ────────────────────────────────────────────────
submission = pd.DataFrame({
    'Index':  test_index,
    'demand': test_preds
})

# Sort by Index to match expected submission order
submission = submission.sort_values('Index').reset_index(drop=True)

# ── Export ─────────────────────────────────────────────────────────────────────
out_file = 'catboost_momentum_v2.csv'
submission.to_csv(out_file, index=False)

print(f"📤 Submission exported → {out_file}")
print(f"   Rows       : {len(submission)}")
print(f"   Demand min : {test_preds.min():.6f}")
print(f"   Demand max : {test_preds.max():.6f}")
print(f"   Demand mean: {test_preds.mean():.6f}")
print(f"   Demand std : {test_preds.std():.6f}")
print(f"\n   First 5 rows:")
print(submission.head().to_string(index=False))
print(f"\n✅ Done — submission ready for upload")